# List expressions

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*J. A. Hernando Morata*. Original material (2020) in collaboration with
*G. Martínez-Lema* and *M. Kekic*.

**Objectives**

* apply a function to every item, or keep some items: `map`, `filter`, and their
  **comprehensions**;
* walk several lists at once with `zip` and `enumerate`;
* build lists, dicts and sets in one expression;
* sort, **reduce**, count and group data.

In [ ]:
import math
import random
import numpy as np

## 1. The functional style

In [Functions](./about_functions.ipynb) we met the two ideas of functional programming:

* **pure** functions: same input, same output, no side effects;
* functions as **objects**: passed, returned, frozen, composed.

A functional program is a flow of data through pure functions. The workhorses of that
flow are three operations on collections of data:

| Operation | Meaning | Math |
|:--|:--|:--|
| **map** | apply a function to every item | $x_i \to f(x_i)$ |
| **filter** | keep the items that satisfy a condition | $\{x_i \mid c(x_i)\}$ |
| **reduce** | combine all the items into one value | $\sum_i x_i$, $\max_i x_i$ |

Python offers each of them twice: as a **function** that takes a function, and as a
**comprehension**, a piece of syntax. We see them in pairs.

## 2. map, filter and comprehensions

### 2.1 map

`map(f, xs)` applies `f` to every item of `xs`. The function is often a `lambda`:

In [ ]:
energies = [10.2, 55.0, 3.1, 120.7]          # GeV

in_mev = list(map(lambda e: 1000. * e, energies))
print(in_mev)

The same with a **list comprehension**, `[expression for item in iterable]`:

In [ ]:
in_mev = [1000. * e for e in energies]
print(in_mev)

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Without list()</b></p>

**Decide**: what does `print(map(lambda e: 1000. * e, energies))` show?

</div>

In [ ]:
print(map(lambda e: 1000. * e, energies))

<details>
<summary><b>Solution</b></summary>

Not the numbers: `<map object at 0x...>`. `map` does not compute anything yet: it returns
an **iterator** that computes each item when asked. `list(...)` asks for all of them.
This laziness is the subject of [Iterators](./about_iterators.ipynb).

</details>

### 2.2 filter

`filter(c, xs)` keeps the items for which `c` is true. As a comprehension, an `if` at the
end:

In [ ]:
high = list(filter(lambda e: e > 50., energies))
print(high)

high = [e for e in energies if e > 50.]
print(high)

### 2.3 Both at once

A comprehension maps and filters in one expression. It reads almost like mathematics:
$\{ 1000\,e \mid e \in E,\; e > 50 \}$.

In [ ]:
print([1000. * e for e in energies if e > 50.])
print(list(map(lambda e: 1000. * e, filter(lambda e: e > 50., energies))))

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Three ways to say the same thing</b></p>

```python
high_mev = []
for e in energies:
    if e > 50.:
        high_mev.append(1000. * e)

high_mev = list(map(lambda e: 1000. * e, filter(lambda e: e > 50., energies)))

high_mev = [1000. * e for e in energies if e > 50.]
```

Which one is the most readable? Which one would you use? Is there a case where `map`
is the better choice?

</div>

<details>
<summary><b>Solution</b></summary>

Most Python programmers prefer the comprehension: one line, read left to right, no
`lambda`. The loop is fine when the body is long. `map` reads well when the function
already has a name: `list(map(math.sqrt, xs))` is as clear as `[math.sqrt(x) for x in xs]`.
What to avoid is `map` + `filter` + two `lambda`s: correct, and hard to read.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Pairs</b></p>

Write each of these twice, with `map`/`filter` and with a comprehension, and check with
`assert` that both give the same:

1. the lengths of the words in `['proton', 'pi', 'kaon', 'lambda']`;
2. the masses (in MeV) below 200 in `[0.511, 105.66, 139.57, 493.68, 938.27]`;
3. the squares of the odd numbers below 20.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
words = ['proton', 'pi', 'kaon', 'lambda']
assert list(map(len, words)) == [len(w) for w in words]

masses = [0.511, 105.66, 139.57, 493.68, 938.27]
assert list(filter(lambda m: m < 200., masses)) == [m for m in masses if m < 200.]

odd_squares = [n * n for n in range(20) if n % 2 == 1]
assert list(map(lambda n: n * n, filter(lambda n: n % 2 == 1, range(20)))) == odd_squares
```

In (1) `map(len, words)` needs no `lambda`: `len` is already a function.

</details>

## 3. Walking several lists: zip and enumerate

### 3.1 zip

`zip` walks several iterables **at the same time**, yielding tuples with one item from
each. In a comprehension, it is the way to combine two lists item by item:

In [ ]:
names = ['e', 'mu', 'tau']
masses = [0.511, 105.66, 1776.9]       # MeV

for name, mass in zip(names, masses):
    print(f'{name:>4} {mass:8.2f} MeV')

print([f'{n}: {m / 1000:.4f} GeV' for n, m in zip(names, masses)])

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Lists of different lengths</b></p>

What does `list(zip([1, 2, 3], ['a', 'b']))` give? Is that a feature or a danger?

</div>

In [ ]:
print(list(zip([1, 2, 3], ['a', 'b'])))

<details>
<summary><b>Solution</b></summary>

`[(1, 'a'), (2, 'b')]`: `zip` stops at the shortest iterable and the `3` is lost, with no
warning. If the lists *must* have the same length (masses and names of particles, `x` and
`y` of a measurement), say it: `zip(xs, ys, strict=True)` raises a `ValueError` when the
lengths differ (Python 3.10+).

</details>

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Unzip</b></p>

`pairs = [(1, 'a'), (2, 'b'), (3, 'c')]`. **Decide** what `list(zip(*pairs))` gives.

</div>

In [ ]:
pairs = [(1, 'a'), (2, 'b'), (3, 'c')]
print(list(zip(*pairs)))

numbers, letters = zip(*pairs)
first, *rest = numbers
print(numbers, letters, first, rest)

<details>
<summary><b>Solution</b></summary>

`[(1, 2, 3), ('a', 'b', 'c')]`. `*pairs` passes the three tuples as three arguments, and
`zip` takes their first items, then their second items: it **transposes**. The last lines
show *starred unpacking*: `first, *rest = xs` puts the first item in `first` and the rest
in a list.

</details>

### 3.2 enumerate

`enumerate` gives the **position** and the **item**. Use it instead of `range(len(xs))`:

In [ ]:
for i, name in enumerate(names):
    print(i, name)

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Position of the maximum</b></p>

1. Build a dictionary `{name: mass}` from `names` and `masses` with `zip`, in one line.
2. Write a function that returns the **position** and the value of the largest item,
   using `enumerate`, and its test. Then do it again with `max` and a `key`.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
mass_table = dict(zip(names, masses))

def argmax(xs):
    """Return the position and the value of the largest item of xs."""
    imax, xmax = 0, xs[0]
    for i, x in enumerate(xs):
        if x > xmax:
            imax, xmax = i, x
    return imax, xmax

def argmax2(xs):
    return max(enumerate(xs), key=lambda ix: ix[1])

def test_argmax():
    xs = [3, 9, 2, 9, 1]
    assert argmax(xs) == argmax2(xs) == (1, 9)

test_argmax()
```

With ties both return the **first** maximum. `np.argmax` does the same.

</details>

## 4. More comprehensions

### 4.1 Several for

A comprehension can have several `for` and `if`. They are read **in the same order** as
the nested loops:

In [ ]:
xs = [0, 1, 2]
ys = [3, 4, 5]

pairs = [(x, y) for x in xs if x % 2 == 0 for y in ys if y % 2 == 1]
print(pairs)

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Write the loops</b></p>

Write the nested loops that build the same `pairs`, **before** opening the solution.
How many items has `[(x, y) for x in range(10) for y in range(20)]`?

</div>

<details>
<summary><b>Solution</b></summary>

```python
pairs = []
for x in xs:
    if x % 2 == 0:
        for y in ys:
            if y % 2 == 1:
                pairs.append((x, y))
```

`10 × 20 = 200`: two `for` make all the combinations, like a double loop. To walk two
lists **in parallel** you need `zip`, not two `for`.

</details>

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Flatten or nest?</b></p>

`m = [[1, 2], [3, 4]]`. **Decide** what each one gives:

```python
[x for row in m for x in row]
[[10 * x for x in row] for row in m]
```

</div>

<details>
<summary><b>Solution</b></summary>

`[1, 2, 3, 4]`: two `for` in one comprehension **flatten** the matrix.
`[[10, 20], [30, 40]]`: a comprehension **inside** another keeps the structure.

</details>

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — The shared rows</b></p>

We want a 3 × 3 grid of zeros, and then set one cell. **Before running**, what is printed?

```python
grid = [[0] * 3] * 3
grid[0][0] = 1
print(grid)
```

</div>

In [ ]:
grid = [[0] * 3] * 3
grid[0][0] = 1
print(grid)

<details>
<summary><b>Solution</b></summary>

`[[1, 0, 0], [1, 0, 0], [1, 0, 0]]`. `[[0] * 3] * 3` repeats the **same** inner list three
times: three names for one row. A comprehension builds a new row each time:

```python
grid = [[0] * 3 for _ in range(3)]
```

(For numbers, `np.zeros((3, 3))` is the right tool anyway.)

</details>

### 4.2 Dict and set comprehensions

The same syntax with `{}` builds a **dictionary** or a **set**:

In [ ]:
lepton_masses = {'e': 0.511, 'mu': 105.66, 'tau': 1776.9}

in_gev = {name: m / 1000. for name, m in lepton_masses.items()}
light = {name for name, m in lepton_masses.items() if m < 200.}

print(in_gev)
print(light)

### 4.3 Generator expressions

And with `()`, a **generator expression**: it does not build the list, it produces the
items one by one, like `map`. Inside a function call the parentheses can be dropped:

In [ ]:
print(sum(x * x for x in range(10)))

We will see what that means, and why it matters, in [Iterators](./about_iterators.ipynb).

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — A comprehension for its side effect</b></p>

```python
[print(e) for e in energies]
```

It prints the energies. What else does it do? Is it good style?

</div>

In [ ]:
[print(e) for e in energies]

<details>
<summary><b>Solution</b></summary>

It also builds a list of four `None` (the return value of `print`), which is thrown away.
A comprehension is for **building** a collection; to **do** something with each item,
write a `for` loop.

</details>

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Comprehension or numpy?</b></p>

In physics we usually have arrays of numbers. Compare:

```python
xs = list(range(1_000_000))
squares = [x * x for x in xs]

xa = np.arange(1_000_000)
squares = xa * xa
```

Which one is faster, and by how much? Measure it with `%timeit`. When is a comprehension
still the right tool?

</div>

In [ ]:
xs = list(range(1_000_000))
xa = np.arange(1_000_000)
%timeit [x * x for x in xs]
%timeit xa * xa

<details>
<summary><b>Solution</b></summary>

numpy is one or two orders of magnitude faster: the loop runs in compiled code, on
numbers stored contiguously, without creating a Python object per item. Note that numpy
is functional too: `xa * xa` is a `map` over the whole array.

A comprehension is still the right tool when the items are **not numbers** (strings,
tuples, dicts, files), when the list is short, or when the operation has no numpy
equivalent.

</details>

## 5. Sorting

`sorted` returns a **new** sorted list; `reverse=True` sorts in **descending** order.

In [ ]:
xs = [8, 10, 9, 5, 3, 1, 4]
print(sorted(xs))
print(sorted(xs, reverse=True))
print(xs)

`key` is a function applied to each item: the items are sorted by its value.

In [ ]:
def distance_to(x0):
    """Return the function x -> |x - x0|."""
    return lambda x: abs(x - x0)

print(sorted(xs, key=distance_to(5)))

In [ ]:
particles = [('e', -1, 0.511), ('p', 1, 938.27), ('mu', -1, 105.66), ('pi', 1, 139.57)]

# by charge, and by mass inside each charge
print(sorted(particles, key=lambda p: (p[1], p[2])))

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — What went wrong?</b></p>

```python
xs = [3, 1, 2]
xs = xs.sort()
print(xs)
```

</div>

<details>
<summary><b>Solution</b></summary>

It prints `None`. `xs.sort()` sorts the list in place and returns `None`, which is then
assigned to `xs`: the list is lost. Use either `xs.sort()` alone or `xs = sorted(xs)`.
`sorted` is the functional version: it does not modify its argument.

</details>

***[+] Lookout.*** Before `key` existed, sorting used a *comparison* function `cmp(x, y)` returning
`-1, 0, 1`. It still works through `functools.cmp_to_key`, but a `key` is simpler and
faster. Python's sort is **stable**: equal items keep their order, so you can also sort in
several passes (first by the secondary key, then by the primary one).

## 6. Reducing

Some built-ins **reduce** an iterable to one value: `sum`, `max`, `min`, and for booleans
`any` (is one of them true?) and `all` (are all of them true?).

In [ ]:
xs = [3, 9, 2, 7]
print(sum(xs), max(xs), min(xs))
print(any(x > 8 for x in xs), all(x > 1 for x in xs))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — The empty list</b></p>

**Decide**: what do `sum([])`, `any([])`, `all([])` and `max([])` give?

</div>

In [ ]:
print(sum([]), any([]), all([]))

In [ ]:
max([])

<details>
<summary><b>Solution</b></summary>

`0`, `False` and `True`; `max([])` raises a `ValueError`. Each result is the **neutral
element** of the operation: adding nothing gives 0; "some item is true" is false if there
are no items; "all items are true" cannot be contradicted by an empty list. The maximum
of nothing has no sensible value, so `max` fails (or use `max(xs, default=...)`).

</details>

### 6.1 reduce

`functools.reduce(f, xs)` folds the list with any function of two arguments:
`f(f(f(x0, x1), x2), x3)`. The module `operator` has the operators as functions:
`add`, `sub`, `mul`, `truediv`, `pow`, `eq`, `lt`… and `itemgetter(i)`, which extracts
item `i` (a ready-made `key` for sorting).

In [ ]:
from functools import reduce
from operator import mul, itemgetter

print(reduce(mul, range(1, 6)))                      # 5!
print(sorted(particles, key=itemgetter(2)))          # by mass, without lambda

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Map, filter, reduce</b></p>

**Write**, before running anything, one expression with `map`/`filter` and `reduce` (or a
comprehension and `sum`) for: the total energy, in MeV, of the events above 50 GeV in
`energies`.

</div>

<details>
<summary><b>Solution</b></summary>

```python
from operator import add

total = reduce(add, map(lambda e: 1000. * e, filter(lambda e: e > 50., energies)))
total = sum(1000. * e for e in energies if e > 50.)
```

The three operations of the functional style in one line. The second form is the
idiomatic one.

</details>

In [ ]:
# your code here

## 7. Counting and grouping

Two collections of the standard library do what we would otherwise write by hand with a
loop and a dictionary:

* `Counter` counts how many times each item appears;
* `defaultdict(list)` groups items by a key.

In [ ]:
from collections import Counter, defaultdict

decays = ['mu nu', 'e nu', 'mu nu', 'mu nu', 'pi0 e nu', 'mu nu', 'e nu']
counts = Counter(decays)
print(counts)
print(counts.most_common(2))

In [ ]:
events = [(1, 52.3), (2, 10.1), (1, 33.0), (3, 71.2), (2, 45.5), (1, 12.9)]   # (run, E)

by_run = defaultdict(list)
for run, energy in events:
    by_run[run].append(energy)

print(dict(by_run))
print({run: sum(es) / len(es) for run, es in by_run.items()})

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Events per run</b></p>

With the `events.csv` of the [UNIX](./fca_unix.ipynb) chapter in mind, generate here 500
events `(run, energy)` with run in 1, 2, 3 and an exponential energy of mean 50 GeV.

1. How many events per run? (`Counter`)
2. The mean energy per run. (`defaultdict` and a dict comprehension)
3. The fraction of events above 100 GeV in each run.

Compare 1 with what `cut -d, -f1 events.csv | sort | uniq -c` would give in the shell.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
rng = random.Random(1)
events = [(rng.choice([1, 2, 3]), rng.expovariate(1 / 50.)) for _ in range(500)]

per_run = Counter(run for run, _ in events)

by_run = defaultdict(list)
for run, energy in events:
    by_run[run].append(energy)

mean_energy = {run: sum(es) / len(es) for run, es in by_run.items()}
high_fraction = {run: sum(1 for e in es if e > 100.) / len(es) for run, es in by_run.items()}
print(per_run, mean_energy, high_fraction, sep='\n')
```

The fraction above 100 GeV should be about $e^{-2} \approx 0.135$ in every run.

</details>

***[+] Lookout.*** This is exactly what `pandas` does with columns: `df.groupby('run')['energy'].mean()`.
And `collections.namedtuple` gives names to the fields of a tuple (`p.mass` instead of
`p[2]`), a first step towards the classes of the next part.

## 8. Practice and review

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A particle table</b></p>

The next cell has a small table of particles: masses in MeV, lifetimes in s.
Using comprehensions, `sorted`, `min`/`max` and `key`, with a test for each answer:

1. the names of the charged particles;
2. the table sorted by mass;
3. a dictionary `{name: mass}`;
4. the lightest **unstable** particle;
5. the particles that fly more than 1 m on average at $\gamma\beta = 1$
   ($c\tau > 1$ m).

</div>

In [ ]:
table = [
    {'name': 'e',    'mass': 0.511,   'charge': -1, 'lifetime': math.inf},
    {'name': 'mu',   'mass': 105.66,  'charge': -1, 'lifetime': 2.197e-6},
    {'name': 'pi+',  'mass': 139.57,  'charge': +1, 'lifetime': 2.603e-8},
    {'name': 'pi0',  'mass': 134.98,  'charge': 0,  'lifetime': 8.5e-17},
    {'name': 'K+',   'mass': 493.68,  'charge': +1, 'lifetime': 1.238e-8},
    {'name': 'p',    'mass': 938.27,  'charge': +1, 'lifetime': math.inf},
    {'name': 'n',    'mass': 939.57,  'charge': 0,  'lifetime': 878.4},
]

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
C = 299792458.   # m/s

charged = [p['name'] for p in table if p['charge'] != 0]
by_mass = sorted(table, key=lambda p: p['mass'])
mass_of = {p['name']: p['mass'] for p in table}
lightest_unstable = min((p for p in table if p['lifetime'] < math.inf),
                        key=lambda p: p['mass'])
long_lived = [p['name'] for p in table if C * p['lifetime'] > 1.]

assert charged == ['e', 'mu', 'pi+', 'K+', 'p']
assert [p['name'] for p in by_mass][:3] == ['e', 'mu', 'pi0']
assert lightest_unstable['name'] == 'mu'
assert long_lived == ['e', 'mu', 'pi+', 'K+', 'p', 'n']
```

The pions and kaons fly metres: that is why they reach the detectors. The $\pi^0$ does
not: it decays at the vertex.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — Review this analysis</b></p>

Rewrite this code in the functional style: no indices, no `append` in a loop where a
comprehension does the job, no hidden state. Keep the result identical, and check it
with an `assert`.

```python
xs = [1.2, -0.5, 3.3, 0.8, -2.1]
ys = [0.4, 1.1, -0.7, 2.2, 0.9]
r = []
n = 0
for i in range(len(xs)):
    if xs[i] > 0 and ys[i] > 0:
        r.append((xs[i] ** 2 + ys[i] ** 2) ** 0.5)
        n = n + 1
mean_r = sum(r) / n
```

</div>

In [ ]:
xs = [1.2, -0.5, 3.3, 0.8, -2.1]
ys = [0.4, 1.1, -0.7, 2.2, 0.9]
r = []
n = 0
for i in range(len(xs)):
    if xs[i] > 0 and ys[i] > 0:
        r.append((xs[i] ** 2 + ys[i] ** 2) ** 0.5)
        n = n + 1
mean_r = sum(r) / n

# your version here

<details>
<summary><b>Solution</b></summary>

```python
mean_r_old = mean_r

radii = [math.hypot(x, y) for x, y in zip(xs, ys) if x > 0 and y > 0]
mean_r = sum(radii) / len(radii)

assert math.isclose(mean_r, mean_r_old)     # hypot may differ in the last bit
```

`zip` instead of `range(len(...))`; a comprehension instead of `append`; `len` instead of
a counter that must be kept in sync by hand; `math.hypot` instead of the formula. Fewer
names, fewer places for a bug.

</details>

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — How would you do it?</b></p>

The fraction of the events in `energies` above 50 GeV. Write it in **four** ways: a loop,
a comprehension, `filter`, and numpy. Which one is the clearest? The fastest for
$10^7$ events? Which one would you write in an exam, on paper?

</div>

<details>
<summary><b>Solution</b></summary>

```python
n_high = 0
for e in energies:
    if e > 50.:
        n_high += 1
f1 = n_high / len(energies)

f2 = sum(e > 50. for e in energies) / len(energies)
f3 = len(list(filter(lambda e: e > 50., energies))) / len(energies)
f4 = np.mean(np.array(energies) > 50.)
```

`f2` is the clearest in plain Python: `sum` counts `True` as 1, without building a list.
`f4` is the fastest by far **if the data are already an array** (converting a list of
$10^7$ floats costs about as much as the loop), and it is also the shortest: the mean of a
boolean array is a fraction.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (in pairs) — Points in the plane</b></p>

Generate $n$ points $(x, y)$ with $x, y$ normally distributed, mean 0 and $\sigma = 1$
(`random.gauss`). Split the work in the group:

* **A**: the distance of each point to the origin;
* **B**: the points sorted by distance to the origin;
* **C**: the points in the first quadrant at a distance larger than 1.

Use only comprehensions, built-ins and `math`. Then each member writes the same with numpy and
checks a teammate's result, with `assert`.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
n = 1000
points = [(random.gauss(0., 1.), random.gauss(0., 1.)) for _ in range(n)]

distances = [math.hypot(x, y) for x, y in points]                       # A
by_distance = sorted(points, key=lambda p: math.hypot(*p))              # B
selected = [(x, y) for x, y in points
            if x > 0 and y > 0 and math.hypot(x, y) > 1]                 # C

# check with numpy
pa = np.array(points)
r = np.hypot(pa[:, 0], pa[:, 1])
assert np.allclose(distances, r)
assert np.allclose(by_distance, pa[np.argsort(r)])
mask = (pa[:, 0] > 0) & (pa[:, 1] > 0) & (r > 1)
assert len(selected) == mask.sum()
```

The fraction selected should be about $e^{-1/2}/4 \approx 0.15$: why?

</details>

## 9. Summary

* `map` ↔ `[f(x) for x in xs]`; `filter` ↔ `[x for x in xs if c(x)]`; reduce ↔ `sum`,
  `max`, `reduce`.
* `zip` walks in parallel, `enumerate` gives the position: no `range(len(...))`.
* `sorted(xs, key=f)` returns a new list; `xs.sort()` sorts in place and returns `None`.
* Count with `Counter`, group with `defaultdict`.

<details>
<summary><b>[i] Cheat sheet — List expressions</b></summary>

| | |
|:--|:--|
| `list(map(f, xs))` / `[f(x) for x in xs]` | apply `f` to every item |
| `list(filter(c, xs))` / `[x for x in xs if c(x)]` | keep the items where `c` is true |
| `{k: v for k, v in d.items()}` | dict comprehension |
| `{f(x) for x in xs}` | set comprehension |
| `sum(f(x) for x in xs)` | generator expression |
| `zip(xs, ys, strict=True)`, `zip(*pairs)` | walk in parallel; transpose |
| `first, *rest = xs` | starred unpacking |
| `enumerate(xs)` | position and item |
| `sorted(xs, key=f, reverse=True)` | new sorted list; `xs.sort()` sorts in place and returns `None` |
| `sum`, `max`, `min`, `any`, `all`, `reduce` | reductions; `max(xs, default=v)` |
| `Counter(xs)`, `defaultdict(list)` | count; group |

**Traps**: `[[0] * 3] * 3` shares the rows; `xs = xs.sort()`; `zip` silently truncates;
a comprehension used only for its side effects.

</details>

### Check yourself

1. Write `list(filter(lambda x: x > 0, xs))` as a comprehension.
2. What does `zip` do with lists of different lengths?
3. How many items does `[(x, y) for x in range(3) for y in range(4)]` have? And
   `[(x, y) for x, y in zip(range(3), range(4))]`?
4. What does `all([])` return, and why?
5. Why is `[[0] * 3] * 3` a trap?

<details>
<summary><b>Solutions</b></summary>

1. `[x for x in xs if x > 0]`.
2. It stops at the shortest one, silently, unless `strict=True`.
3. 12 (all the combinations) and 3 (in parallel, up to the shortest).
4. `True`: no item contradicts "all are true" (the neutral element of `and`).
5. The three rows are the same list: changing one changes all.

</details>

### Bibliography

1. H. Abelson, G. J. Sussman and J. Sussman, *Structure and Interpretation of Computer
   Programs*, 2nd ed., MIT Press (1996).
   [Online](https://mitpress.mit.edu/sites/default/files/sicp/full-text/book/book.html).
2. [Functional Programming HOWTO](https://docs.python.org/3/howto/functional.html),
   Python documentation.